# MCD-rPPG Dataset: Training Notebook

## Overview
This notebook prepares and trains a model on the processed MCD-rPPG dataset.

### Workflow:
1. **Data Inspection**: Load and check format of NPZ files
2. **Visualization**: Plot ROI, PPG, and time delta from sample files
3. **Vitals Display**: Print vital signs from metadata
4. **Model Adaptation**: Adapt SCNN_8rois.py for our ROI structure
5. **Training**: Train the model on processed data

### Dataset Structure:
- NPZ files with ROI data, PPG values, time deltas, landmarks, metadata, vitals
- 9 ROIs: full_face, forehead, left_eye, right_eye, nose, mouth, chin, right_cheek_50, left_cheek_280, chin_199
- ROI box size: 24x24 pixels
- Chunk size: 450 frames with 150-frame overlap

### Model:
- Adapted from rppglib/models/SCNN_8rois.py
- Modified for 9 ROIs (original had 8)
- Input: ROI sequences, Output: PPG signal


In [ ]:
# Install required packages
!pip install torch torchvision torchaudio -q
!pip install numpy pandas matplotlib seaborn tqdm -q
!pip install scikit-learn -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')

# Configuration
CHUNKS_PATH = '/home/cristic/preprocessed_data/chunks/'
OUTPUT_PATH = '/home/cristic/training_output/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ROI Configuration (must match processing pipeline)
ROI_NAMES = [
    'full_face', 'forehead', 'left_eye', 'right_eye',
    'nose', 'mouth', 'chin', 'right_cheek_50',
    'left_cheek_280', 'chin_199'
]
NUM_ROIS = len(ROI_NAMES)
ROI_BOX_SIZE = (24, 24)

# Training parameters
BATCH_SIZE = 8
NUM_EPOCHS = 50
LEARNING_RATE = 0.001
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Configuration:')
print(f'  Chunks path: {CHUNKS_PATH}')
print(f'  Number of ROIs: {NUM_ROIS}')
print(f'  ROI names: {ROI_NAMES}')
print(f'  Device: {DEVICE}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Epochs: {NUM_EPOCHS}')
print(f'  Learning rate: {LEARNING_RATE}')

## 1. Data Inspection: Load and Check NPZ Files

First, let's load a few NPZ files and inspect their structure to understand the data format.

In [ ]:
# Find all NPZ files
npz_files = []
for root, dirs, files in os.walk(CHUNKS_PATH):
    for file in files:
        if file.endswith('.npz'):
            npz_files.append(os.path.join(root, file))

npz_files = sorted(npz_files)
print(f'Found {len(npz_files)} NPZ files')
print()

# Load first few files for inspection
sample_files = npz_files[:5]
print('Inspecting first 5 files:')
print('-' * 80)

file_info = []
for filepath in sample_files:
    basename = os.path.basename(filepath)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    
    with np.load(filepath) as data:
        roi_shapes = {}
        for key in data.files:
            if key.startswith('roi_'):
                roi_name = key[4:]
                roi_shapes[roi_name] = data[key].shape
        
        ppg_shape = data.get('ppg_values', np.array([])).shape
        time_deltas_shape = data.get('time_deltas', np.array([])).shape
        landmarks_shape = data.get('landmarks', np.array([])).shape
        
        file_info.append({
            'filename': basename,
            'size_mb': size_mb,
            'num_frames': roi_shapes.get('full_face', (0,))[0] if roi_shapes else 0,
            'num_rois': len(roi_shapes),
            'roi_shape': roi_shapes.get('full_face', (0, 0, 0, 0)),
            'ppg_shape': ppg_shape,
            'time_deltas_shape': time_deltas_shape,
            'landmarks_shape': landmarks_shape
        })
        
        print(f'File: {basename} ({size_mb:.2f} MB)')
        print(f'  Frames: {roi_shapes.get("full_face", (0,))[0] if roi_shapes else 0}')
        print(f'  ROIs: {len(roi_shapes)}')
        print(f'  ROI shape: {roi_shapes.get("full_face", (0, 0, 0, 0))}')
        print(f'  PPG shape: {ppg_shape}')
        print(f'  Time deltas shape: {time_deltas_shape}')
        print(f'  Landmarks shape: {landmarks_shape}')
        print()

# Display summary
if file_info:
    info_df = pd.DataFrame(file_info)
    print('Summary:')
    print('-' * 80)
    display(info_df[['filename', 'size_mb', 'num_frames', 'num_rois']])

## 2. Visualization: Plot ROI, PPG, and Time Delta

Let's load one NPZ file and visualize the data to understand the format.

In [ ]:
# Load first file for visualization
if npz_files:
    sample_file = npz_files[0]
    print(f'Loading: {os.path.basename(sample_file)}')
    
    with np.load(sample_file) as data:
        # Load ROI data
        roi_data = {}
        for key in data.files:
            if key.startswith('roi_'):
                roi_name = key[4:]
                roi_data[roi_name] = data[key]
        
        # Load other data
        ppg_values = data.get('ppg_values', np.array([]))
        time_deltas = data.get('time_deltas', np.array([]))
        landmarks = data.get('landmarks', np.array([]))
        
        # Load metadata
        metadata = {}
        for key in data.files:
            if key.startswith('meta_'):
                metadata[key[5:]] = data[key]
        
        # Load vital signs
        vital_signs = {}
        for key in data.files:
            if key.startswith('vital_'):
                vital_signs[key[6:]] = data[key]
    
    print(f'Loaded data:')
    print(f'  Number of frames: {len(next(iter(roi_data.values()))) if roi_data else 0}')
    print(f'  Number of ROIs: {len(roi_data)}')
    print(f'  PPG values: {len(ppg_values)}')
    print(f'  Time deltas: {len(time_deltas)}')
    print(f'  Landmarks: {landmarks.shape if landmarks.size > 0 else 0}')
    print()
    
    # Plot ROI samples from first, middle, and last frames
    print('=' * 80)
    print('ROI SAMPLES FROM FIRST, MIDDLE, AND LAST FRAMES')
    print('=' * 80)
    
    n_rois = len(roi_data)
    n_frames = len(next(iter(roi_data.values())))
    frame_indices = [0, n_frames // 2, n_frames - 1]
    frame_labels = ['First Frame', 'Middle Frame', 'Last Frame']
    
    # Plot each frame position
    for frame_idx, label in zip(frame_indices, frame_labels):
        fig, axes = plt.subplots(3, 4, figsize=(16, 12))
        axes = axes.ravel()
        
        print(f'\n{label}:')
        
        for idx, (roi_name, roi_frames) in enumerate(roi_data.items()):
            if idx < len(axes):
                frame = roi_frames[frame_idx]
                axes[idx].imshow(frame)
                axes[idx].set_title(f'{roi_name}\n{frame.shape}', fontsize=10)
                axes[idx].axis('off')
        
        for idx in range(len(roi_data), len(axes)):
            axes[idx].axis('off')
        
        plt.suptitle(f'{label} - ROI Samples', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    print()
    
    # Plot PPG signal
    print('=' * 80)
    print('PPG SIGNAL')
    print('=' * 80)
    
    if len(ppg_values) > 0:
        fig, ax = plt.subplots(1, 1, figsize=(15, 4))
        ax.plot(ppg_values, color='red', linewidth=1.5)
        ax.set_title('PPG Signal', fontsize=14, fontweight='bold')
        ax.set_xlabel('Frame Index')
        ax.set_ylabel('PPG Value')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f'PPG Statistics:')
        print(f'  Min: {ppg_values.min():.4f}')
        print(f'  Max: {ppg_values.max():.4f}')
        print(f'  Mean: {ppg_values.mean():.4f}')
        print(f'  Std: {ppg_values.std():.4f}')
        print()
    
    # Plot time deltas
    print('=' * 80)
    print('TIME DELTA (FRAME INTERVALS)')
    print('=' * 80)
    
    if len(time_deltas) > 0:
        fig, ax = plt.subplots(1, 1, figsize=(15, 4))
        ax.plot(time_deltas, color='blue', linewidth=1.5)
        ax.set_title('Time Deltas (Frame-to-Frame Intervals)', fontsize=14, fontweight='bold')
        ax.set_xlabel('Frame Index')
        ax.set_ylabel('Time Delta (seconds)')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f'Time Delta Statistics:')
        print(f'  Min: {time_deltas.min():.6f}')
        print(f'  Max: {time_deltas.max():.6f}')
        print(f'  Mean: {time_deltas.mean():.6f}')
        print(f'  Std: {time_deltas.std():.6f}')
        print()
    
    # Print metadata
    print('=' * 80)
    print('METADATA')
    print('=' * 80)
    for key, value in metadata.items():
        print(f'  {key}: {value}')
    print()
    
    # Print vital signs
    print('=' * 80)
    print('VITAL SIGNS')
    print('=' * 80)
    if vital_signs:
        for key, value in vital_signs.items():
            print(f'  {key}: {value}')
    else:
        print('  No vital signs available')
    print()

else:
    print('No NPZ files found')

## 3. Data Preparation for Training

Now let's prepare the data for training. We need to:
- Load all NPZ files
- Extract ROI sequences and PPG signals
- Create PyTorch Dataset and DataLoader
- Normalize data


In [ ]:
class MCDrPPGDataset(Dataset):
    '''PyTorch Dataset for MCD-rPPG processed data.'''
    
    def __init__(self, npz_files, roi_names=None, window_size=64, stride=16, normalize=True):
        self.npz_files = npz_files
        self.roi_names = roi_names or ROI_NAMES
        self.window_size = window_size
        self.stride = stride
        self.normalize = normalize
        
        # Load all data
        self.data = self._load_all_data()
        
        print(f'Loaded {len(self.data["roi_sequences"])} sequences from {len(npz_files)} files')
    
    def _load_all_data(self):
        roi_sequences = []
        ppg_sequences = []
        metadata_list = []
        
        for filepath in tqdm(self.npz_files, desc='Loading NPZ files'):
            with np.load(filepath) as data:
                # Load ROI data
                roi_data = {}
                for key in data.files:
                    if key.startswith('roi_'):
                        roi_name = key[4:]
                        if roi_name in self.roi_names:
                            roi_data[roi_name] = data[key]
                
                # Load PPG data
                ppg_values = data.get('ppg_values', np.array([]))
                time_deltas = data.get('time_deltas', np.array([]))
                
                # Check if we have data
                if not roi_data or len(ppg_values) == 0:
                    continue
                
                # Get number of frames
                first_roi = list(roi_data.values())[0]
                n_frames = first_roi.shape[0]
                
                # Create sequences using sliding window
                for start in range(0, n_frames - self.window_size + 1, self.stride):
                    end = start + self.window_size
                    
                    # Extract ROI sequence for this window
                    roi_seq = []
                    for roi_name in self.roi_names:
                        if roi_name in roi_data:
                            roi_frames = roi_data[roi_name]
                            window_frames = roi_frames[start:end]
                            roi_seq.append(window_frames)
                    
                    # Stack ROIs: (window_size, num_rois, height, width, channels)
                    roi_seq = np.stack(roi_seq, axis=1)  # Shape: (T, N, H, W, C)
                    
                    # Extract PPG sequence
                    ppg_seq = ppg_values[start:end]
                    
                    # Normalize if requested
                    if self.normalize:
                        roi_seq = roi_seq.astype(np.float32) / 255.0
                        ppg_seq = (ppg_seq - ppg_seq.mean()) / (ppg_seq.std() + 1e-6)
                    
                    roi_sequences.append(roi_seq)
                    ppg_sequences.append(ppg_seq)
                    
                    # Store metadata
                    metadata_list.append({
                        'file': os.path.basename(filepath),
                        'start_frame': start,
                        'end_frame': end,
                        'window_size': self.window_size
                    })
        
        return {
            'roi_sequences': np.array(roi_sequences),
            'ppg_sequences': np.array(ppg_sequences),
            'metadata': metadata_list
        }
    
    def __len__(self):
        return len(self.data['roi_sequences'])
    
    def __getitem__(self, idx):
        roi_seq = self.data['roi_sequences'][idx]
        ppg_seq = self.data['ppg_sequences'][idx]
        
        # Convert to PyTorch tensors
        # roi_seq shape: (T, N, H, W, C) -> (N, T, H, W, C) for model
        roi_tensor = torch.from_numpy(roi_seq).float().permute(1, 0, 2, 3, 4)
        ppg_tensor = torch.from_numpy(ppg_seq).float()
        
        return roi_tensor, ppg_tensor
    
    def get_shapes(self):
        if len(self.data['roi_sequences']) > 0:
            roi_shape = self.data['roi_sequences'][0].shape
            ppg_shape = self.data['ppg_sequences'][0].shape
            return roi_shape, ppg_shape
        return (0,), (0,)

print('MCDrPPGDataset class defined')

In [ ]:
# Load and prepare data
print('=' * 80)
print('LOADING AND PREPARING DATA')
print('=' * 80)

# Use all NPZ files
print(f'Loading {len(npz_files)} NPZ files...')

# Create dataset
dataset = MCDrPPGDataset(
    npz_files=npz_files,
    roi_names=ROI_NAMES,
    window_size=64,
    stride=16,
    normalize=True
)

# Get shapes
roi_shape, ppg_shape = dataset.get_shapes()
print(f'\nDataset Information:')
print(f'  Total sequences: {len(dataset)}')
print(f'  ROI shape: {roi_shape}')
print(f'  PPG shape: {ppg_shape}')
print(f'  Window size: {roi_shape[0]} frames')
print(f'  Number of ROIs: {roi_shape[1]}')
print(f'  ROI dimensions: {roi_shape[2]}x{roi_shape[3]}x{roi_shape[4]}')
print()

# Split into train and test
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

print(f'Train size: {len(train_dataset)}')
print(f'Test size: {len(test_dataset)}')
print()

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

print(f'Train loader: {len(train_loader)} batches')
print(f'Test loader: {len(test_loader)} batches')
print()

# Test a batch
print('Testing a batch from train loader...')
sample_roi, sample_ppg = next(iter(train_loader))
print(f'Batch ROI shape: {sample_roi.shape}')
print(f'Batch PPG shape: {sample_ppg.shape}')
print(f'ROI dtype: {sample_roi.dtype}')
print(f'PPG dtype: {sample_ppg.dtype}')
print()
print('=' * 80)
print('DATA PREPARATION COMPLETE')
print('=' * 80)

## 4. Model Adaptation: Adapt SCNN_8rois.py for 9 ROIs

Now let's adapt the SCNN_8rois model from rppglib for our 9 ROI structure.

In [ ]:
class Conv2dBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel, stride=None, padding=None):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel, stride=stride, bias=False, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.act(x)
        return x

class Conv1dBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel, stride=None, padding=None):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel, stride=stride, bias=False, padding=padding)
        self.bn = nn.BatchNorm1d(out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.act(x)
        return x

class InvBottleneckBlock1d(nn.Module):
    def __init__(self, channels, kernel, mult=4):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels*mult, kernel, stride=1, bias=False, padding=kernel//2)
        self.norm1 = nn.BatchNorm1d(channels*mult)
        self.act1 = nn.GELU()
        self.conv2 = nn.Conv1d(channels*mult, channels, kernel, stride=1, bias=False, padding=kernel//2)
        self.norm2 = nn.BatchNorm1d(channels)

    def forward(self, x):
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.act1(x)
        x = self.conv2(x)
        x = self.norm2(x)
        return x

class ResBlock1d(nn.Module):
    def __init__(self, channels, kernel, mult=4):
        super().__init__()
        self.block = InvBottleneckBlock1d(channels, kernel, mult=mult)
        self.act = nn.GELU()

    def forward(self, x):
        x = x + self.block(x)
        x = self.act(x)
        return x

class Conv1dBlockRes(nn.Module):
    def __init__(self, in_channels, out_channels, kernel, stride=None, padding=None):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel, stride=stride, bias=False, padding=padding)
        self.bn = nn.BatchNorm1d(out_channels)
        self.act = nn.GELU()
        self.res_block = ResBlock1d(out_channels, kernel)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.act(x)
        x = self.res_block(x)
        return x


In [ ]:
class SCNN_9rois_NN(nn.Module):
    '''Adapted SCNN model for 9 ROIs.
    
    This model is adapted from rppglib/models/SCNN_8rois.py.
    
    ORIGINAL MODEL EXPECTS: (batch_size, window_size, num_rois)
    - The original uses preprocessed mean RGB values from regions
    - Input: (B, T, N) where N=8 regions
    
    OUR ADAPTATION:
    - We use raw ROI pixel values
    - We need to extract spatial-temporal features from the ROIs
    - Input: (B, N, T, H, W, C) where N=9 ROIs, T=window_size, H=24, W=24, C=3
    
    ARCHITECTURE:
    1. Process each ROI separately to extract temporal features
    2. Combine ROI features
    3. Predict PPG signal
    
    Input: (batch_size, num_rois, window_size, height, width, channels)
        Example: (8, 9, 64, 24, 24, 3)
    
    Output: (batch_size, window_size, 1)
        Example: (8, 64, 1) - predicted PPG for each frame
    '''
    
    def __init__(self, num_rois=NUM_ROIS, window_size=64, height=24, width=24, channels=3):
        super().__init__()
        self.num_rois = num_rois
        self.window_size = window_size
        self.height = height
        self.width = width
        self.channels = channels
        
        # Process each ROI through shared 2D CNN to extract spatial features
        # Input per ROI: (B, T, H, W, C) -> we'll process as (B*T, C, H, W)
        base_channels = 32
        
        # 2D CNN for spatial feature extraction (applied to each ROI)
        self.roi_feature_extractor = nn.Sequential(
            Conv2dBlock(channels, base_channels, (3, 3), stride=(1, 1), padding=(1, 1)),
            Conv2dBlock(base_channels, base_channels, (3, 3), stride=(1, 1), padding=(1, 1)),
            nn.AdaptiveAvgPool2d((1, 1))  # Reduce to single spatial point
        )
        
        # Temporal processing
        self.temporal_conv1 = Conv1dBlockRes(base_channels, base_channels*2, 5, stride=1, padding=2)
        self.temporal_conv2 = Conv1dBlockRes(base_channels*2, base_channels*2, 5, stride=1, padding=2)
        self.temporal_conv3 = Conv1dBlockRes(base_channels*2, base_channels*2, 5, stride=1, padding=2)
        self.temporal_conv4 = Conv1dBlockRes(base_channels*2, base_channels*2, 5, stride=1, padding=2)
        
        # Output layer
        self.output_conv = nn.Conv1d(base_channels*2, 1, 1)
        
        # Attention across ROIs
        self.roi_attention = nn.Sequential(
            nn.Linear(base_channels*2, base_channels*2),
            nn.GELU(),
            nn.Linear(base_channels*2, 1),
            nn.Softmax(dim=1)
        )
    
    def forward(self, x):
        # Input: (B, N, T, H, W, C)
        B, N, T, H, W, C = x.shape
        
        # Reshape to process all ROIs and frames together
        # (B, N, T, H, W, C) -> (B*N*T, C, H, W)
        x = x.permute(0, 1, 2, 5, 3, 4)  # (B, N, T, C, H, W)
        x = x.reshape(B * N * T, C, H, W)
        
        # Extract spatial features for each (ROI, frame) pair
        # Output: (B*N*T, base_channels, 1, 1)
        features = self.roi_feature_extractor(x)
        features = features.squeeze(-1).squeeze(-1)  # (B*N*T, base_channels)
        
        # Reshape back: (B*N*T, base_channels) -> (B*N, T, base_channels)
        features = features.reshape(B * N, T, -1)
        
        # Apply temporal convolutions
        features = self.temporal_conv1(features)
        features = self.temporal_conv2(features)
        features = self.temporal_conv3(features)
        features = self.temporal_conv4(features)
        
        # Apply ROI attention
        # features shape: (B*N, base_channels*2, T)
        features = features.permute(0, 2, 1)  # (B*N, T, base_channels*2)
        
        # Average over time for attention
        roi_features = features.mean(dim=1)  # (B*N, base_channels*2)
        
        # Reshape for attention: (B, N, base_channels*2)
        roi_features = roi_features.reshape(B, N, -1)
        
        # Compute attention weights
        attention_weights = self.roi_attention(roi_features)  # (B, N, 1)
        attention_weights = attention_weights.squeeze(-1)  # (B, N)
        
        # Apply attention to features
        features = features.reshape(B, N, T, -1)  # (B, N, T, base_channels*2)
        features = features * attention_weights.unsqueeze(-1).unsqueeze(-1)
        features = features.sum(dim=1)  # (B, T, base_channels*2)
        
        # Final output
        output = self.output_conv(features.permute(0, 2, 1))  # (B, 1, T)
        output = output.permute(0, 2, 1)  # (B, T, 1)
        
        return output
    
    def get_input_shape(self):
        return (self.num_rois, self.window_size, self.height, self.width, self.channels)
    
    def get_output_shape(self, batch_size):
        return (batch_size, self.window_size, 1)

## 5. Training

Now let's train the model on our prepared data.

In [ ]:
# Training setup
print('=' * 80)
print('TRAINING SETUP')
print('=' * 80)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    verbose=True
)

# Training history
history = {
    'train_loss': [],
    'test_loss': [],
    'learning_rate': []
}

print(f'Training for {NUM_EPOCHS} epochs...')
print(f'Batch size: {BATCH_SIZE}')
print(f'Learning rate: {LEARNING_RATE}')
print(f'Device: {DEVICE}')
print()

# Training loop
best_test_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    train_loss = 0.0
    num_train_batches = 0
    
    for batch_roi, batch_ppg in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} - Training'):
        batch_roi = batch_roi.to(DEVICE)
        batch_ppg = batch_ppg.to(DEVICE)
        
        # Forward pass
        pred_ppg = model(batch_roi)
        
        # Compute loss
        loss = criterion(pred_ppg, batch_ppg.unsqueeze(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        num_train_batches += 1
    
    # Validation
    model.eval()
    test_loss = 0.0
    num_test_batches = 0
    
    with torch.no_grad():
        for batch_roi, batch_ppg in tqdm(test_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} - Validation'):
            batch_roi = batch_roi.to(DEVICE)
            batch_ppg = batch_ppg.to(DEVICE)
            
            pred_ppg = model(batch_roi)
            loss = criterion(pred_ppg, batch_ppg.unsqueeze(-1))
            
            test_loss += loss.item()
            num_test_batches += 1
    
    # Compute average losses
    train_loss /= num_train_batches if num_train_batches > 0 else 1
    test_loss /= num_test_batches if num_test_batches > 0 else 1
    
    # Update learning rate
    scheduler.step(test_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])
    
    # Print epoch statistics
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}:')
    print(f'  Train Loss: {train_loss:.6f}')
    print(f'  Test Loss: {test_loss:.6f}')
    print(f'  Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}')
    print()
    
    # Save best model
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        model_path = os.path.join(OUTPUT_PATH, 'scnn_9rois_best.pth')
        torch.save(model.state_dict(), model_path)
        print(f'Best model saved: {model_path} (test loss: {test_loss:.6f})')
        print()

# Save final model
final_model_path = os.path.join(OUTPUT_PATH, 'scnn_9rois_final.pth')
torch.save(model.state_dict(), final_model_path)
print(f'Final model saved: {final_model_path}')
print()

print('=' * 80)
print('TRAINING COMPLETE')
print('=' * 80)

In [ ]:
# Plot training history
print('=' * 80)
print('TRAINING HISTORY')
print('=' * 80)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Plot losses
ax1.plot(history['train_loss'], label='Train Loss', color='blue', linewidth=2)
ax1.plot(history['test_loss'], label='Test Loss', color='red', linewidth=2)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (MSE)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot learning rate
ax2.plot(history['learning_rate'], label='Learning Rate', color='green', linewidth=2)
ax2.set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning Rate')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Final Results:')
print(f'  Best Test Loss: {best_test_loss:.6f}')
print(f'  Final Train Loss: {history["train_loss"][-1]:.6f}')
print(f'  Final Test Loss: {history["test_loss"][-1]:.6f}')
print()

# Save history
history_df = pd.DataFrame(history)
history_file = os.path.join(OUTPUT_PATH, 'training_history.csv')
history_df.to_csv(history_file, index=False)
print(f'Training history saved to: {history_file}')

print('=' * 80)
print('ALL DONE!')
print('=' * 80)

In [ ]:
# Test inference on a sample
print('=' * 80)
print('INFERENCE TEST')
print('=' * 80)

# Load best model
model_path = os.path.join(OUTPUT_PATH, 'scnn_9rois_best.pth')
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model = model.to(DEVICE)
    print(f'Loaded model from: {model_path}')
else:
    print(f'Model not found at: {model_path}')
    print('Using current model weights')

model.eval()

# Get a sample from test set
sample_roi, sample_ppg = next(iter(test_loader))
sample_roi = sample_roi[:1].to(DEVICE)  # Take first sample from batch
sample_ppg = sample_ppg[:1].to(DEVICE)

print(f'Sample ROI shape: {sample_roi.shape}')
print(f'Sample PPG shape: {sample_ppg.shape}')

# Run inference
with torch.no_grad():
    pred_ppg = model(sample_roi)

print(f'Predicted PPG shape: {pred_ppg.shape}')

# Plot results
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))

# True PPG
ax1.plot(sample_ppg[0].cpu().numpy(), label='True PPG', color='blue', linewidth=2)
ax1.set_title('True PPG Signal', fontsize=14, fontweight='bold')
ax1.set_xlabel('Frame Index')
ax1.set_ylabel('PPG Value')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Predicted PPG
ax2.plot(pred_ppg[0].cpu().numpy(), label='Predicted PPG', color='red', linewidth=2)
ax2.set_title('Predicted PPG Signal', fontsize=14, fontweight='bold')
ax2.set_xlabel('Frame Index')
ax2.set_ylabel('PPG Value')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute MSE
mse = criterion(pred_ppg, sample_ppg.unsqueeze(-1)).item()
print(f'MSE on sample: {mse:.6f}')

print('=' * 80)
print('INFERENCE TEST COMPLETE')
print('=' * 80)

## Summary

### What This Notebook Does:

1. **Data Inspection**: Loads NPZ files and checks their format
2. **Visualization**: Plots ROI samples, PPG signals, and time deltas
3. **Vitals Display**: Shows vital signs from metadata
4. **Data Preparation**: Creates PyTorch Dataset and DataLoader
5. **Model Adaptation**: Adapts SCNN_8rois.py for 9 ROIs
6. **Training**: Trains the model with MSE loss
7. **Inference Test**: Tests the trained model on a sample

### Key Features:

- **9 ROI Support**: Adapted from original 8-ROI model
- **Sliding Window**: Creates sequences from chunks
- **Normalization**: Automatic ROI and PPG normalization
- **Training History**: Saves loss curves and learning rate
- **Model Saving**: Saves best and final models

### Output Files:

- `scnn_9rois_best.pth`: Best model weights
- `scnn_9rois_final.pth`: Final model weights
- `training_history.csv`: Training history (loss, learning rate)

### Next Steps:

- Use the trained model for inference on new data
- Fine-tune hyperparameters (learning rate, batch size, epochs)
- Experiment with different window sizes
- Add more sophisticated loss functions
- Implement heart rate estimation from PPG predictions